# A FAIR² Mental Health Survey Dataset from Kilifi County, Kenya: Demonstrating AI-Ready Data Standards for Data Infrastructure in Africa Exploration with `mlcroissant`

This notebook provides a template for loading and exploring a dataset using the [`mlcroissant`](https://github.com/mlcommons/croissant) library. You will:

- Load and examine metadata from a Croissant schema
- Explore available record sets and field IDs (referenced by their `@id`s)
- Extract records and load them into DataFrames
- Perform basic data processing and visualization

### Dataset Source
The Croissant schema for this dataset is available at:

`https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json`

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading

Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json'

# Load the dataset's metadata
dataset = mlc.Dataset(croissant_url)

metadata = dataset.metadata
print(f"Dataset Name: {metadata.name}")
print(f"Description: {metadata.description}")
print(f"Published: {getattr(metadata, 'datePublished', 'N/A')}")

## 2. Data Overview

Review available record sets, their `@id`s, field details, and linked columns. All references use their Croissant `@id` for reproducibility.

Below, we print the available record sets and their primary fields.

In [ ]:
# List record sets, fields and columns by their @id
record_set_objs = list(metadata.record_sets)
print(f"Number of record sets: {len(record_set_objs)}\n")
record_set_ids = []
for rs in record_set_objs:
    print("Record Set @id:", rs.id_)
    print("  Name:", rs.name)
    print("  Description:", getattr(rs, 'description', None))
    print("  Fields:")
    for field in rs.fields:
        print(f"    - Field @id: {field.id_} | Name: {field.name} | Data type: {field.data_type}")
        # Print linked columns (they also have @id)
        if hasattr(field, 'columns') and field.columns:
            for column in field.columns:
                print(f"      * Column @id: {column.id_} | Name: {column.name}")
    print()
    record_set_ids.append(rs.id_)
print("Record Set IDs found:")
print(record_set_ids)

## 3. Data Extraction

Load data from the main record set into a DataFrame for analysis. Use the record set and field `@id`s from above.

We'll load all record sets into DataFrames for flexibility.

In [ ]:
# Extract records for each record set by @id
dfs = {}
for rs_id in record_set_ids:
    records = list(dataset.records(record_set=rs_id))  # Each record is a dict with field @id as keys
    dfs[rs_id] = pd.DataFrame(records)

# Show the columns (field @ids) for the first record set
main_rs_id = record_set_ids[0]
print(f"Loaded columns (field @ids) for main record set '{main_rs_id}':\n{dfs[main_rs_id].columns.tolist()}")
dfs[main_rs_id].head()

## 4. Exploratory Data Analysis (EDA)

Let's perform EDA using the loaded DataFrame. We'll:
- Select a numeric field (e.g., GAD-7, PHQ-9, or PSQ score — verify the `@id` from above)
- Filter records with score above a threshold
- Normalize the numeric field
- Group by a demographic field (e.g., `level_of_education`, `marital_status`, etc.) for analysis

All operations reference fields by their `@id`.

In [ ]:
# Example field @ids based on FAIR2 schema (adjust as needed from printed output):
# Example numeric field: '@gad7_total_score' (for GAD-7 score)
# Example group field: '@level_of_education' (for education level)
#
# You may need to copy actual field @ids from earlier cells' outputs if different.

main_df = dfs[main_rs_id]

# Example: suppose field @ids for GAD-7 total score and education level:
numeric_field_id = '@gad7_total_score'  # Replace with actual @id printed above
group_field_id = '@level_of_education'  # Replace with actual @id printed above

# Check the presence of the fields:
print(f"Columns in main record set: {main_df.columns.tolist()}")
if numeric_field_id not in main_df.columns:
    raise ValueError(f"Field {numeric_field_id} not found in main DataFrame. Please check actual field @id.")
if group_field_id not in main_df.columns:
    print(f"Note: group field {group_field_id} not found. Grouping will be skipped.")

# Convert numeric field to float for processing (in case it's loaded as string/object)
main_df[numeric_field_id] = pd.to_numeric(main_df[numeric_field_id], errors='coerce')

threshold = 10
filtered_df = main_df[main_df[numeric_field_id] > threshold]
print(f"Filtered records with {numeric_field_id} > {threshold} (n={len(filtered_df)}):")
print(filtered_df[[numeric_field_id]].head())

# Normalize the field (z-score normalization)
filtered_df[f"{numeric_field_id}_normalized"] = (
    filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()
) / filtered_df[numeric_field_id].std()
print(f"Normalized {numeric_field_id} for filtered records:")
print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

# Optional: Group by education level
if group_field_id in filtered_df.columns:
    grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().to_frame('mean_score')
    print(f"Grouped mean {numeric_field_id} by {group_field_id}:")
    print(grouped_df.head())

## 5. Visualization

Visualize the distribution of GAD-7 scores and group means by education level (if applicable).

You can easily adapt this to other numeric or group fields by changing the field `@id` variables.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Histogram of the numeric scores
plt.figure(figsize=(8,4))
sns.histplot(main_df[numeric_field_id].dropna(), bins=20, kde=True)
plt.title(f"Distribution of {numeric_field_id}")
plt.xlabel(numeric_field_id)
plt.ylabel('Count')
plt.show()

# Boxplot of scores by education level (if group present)
if group_field_id in main_df.columns:
    plt.figure(figsize=(10,4))
    sns.boxplot(x=main_df[group_field_id], y=main_df[numeric_field_id])
    plt.title(f"{numeric_field_id} by {group_field_id}")
    plt.xlabel(group_field_id)
    plt.ylabel(numeric_field_id)
    plt.xticks(rotation=45)
    plt.show()

## 6. Conclusion

- We explored a FAIR²-compliant mental health survey dataset using `mlcroissant` by referencing all entities with their Croissant `@id` fields.
- We examined the schema, listed available record sets and fields, and dynamically extracted data for analysis.
- Data was processed and normalized for further study and grouped by demographic fields.
- Visual analysis revealed the distribution and group variations in GAD-7 scores.

To extend this notebook, try analyzing PHQ-9 and PSQ scores or cross-analyzing other demographic fields, always referencing their `@id`s for robustness.